# Figure S4: Projected scHPF Factors on iTF and iMG

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys 
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import scanpy as sc 
import muon as mu

sys.path.append('../utils')

import signature_heatmaps as signature_heatmaps
import factor_labels as factor_labels

## Load Data

In [ ]:
data_dir = "<path to processed data>"

cite_6tf_path = os.path.join(data_dir, "cite_6tf_cleaned_revisions.h5mu")
cite_imgl_path = os.path.join(data_dir, "cite_imgl_cleaned_revisions.h5mu")
merged_6tf_path = os.path.join(data_dir, "adata_revisions_merged_6tf.h5ad")

In [ ]:
mdata_dict = {}
mdata_dict['cite_6tf'] = mu.read_h5mu(cite_6tf_path)
mdata_dict['cite_imgl'] = mu.read_h5mu(cite_imgl_path)

adata_dict = {}
adata_dict['merged_6tf'] = sc.read_h5ad(merged_6tf_path)
adata_dict['cite_6tf'] = mdata_dict['cite_6tf'].mod['rna'].copy()
adata_dict['cite_imgl'] = mdata_dict['cite_imgl'].mod['rna'].copy()

In [ ]:
signature_cols_ordered = ['homeostatic_score_ucell',
 'interferon_score_ucell',
 'chemokine_score_ucell',
 'antigen_presenting_score_ucell',
 'dam_score_ucell',
 'lipid_dam_score_ucell']

## Masking for analysis
to exclude ntc_g5 + foxk1_g2 + mixscale_cutoff >= 0

In [ ]:
guides_to_exclude = ['FOXK1_g2', 'non-targeting_g5']

adata_6tf_clean = adata_dict['merged_6tf'][~adata_dict['merged_6tf'].obs['guide'].isin(["non-targeting_g5", "FOXK1_g2"])]
adata_imgl_clean = adata_dict['cite_imgl'][~adata_dict['cite_imgl'].obs['guide'].isin(["non-targeting_g5", "FOXK1_g2"])]
adata_6tf_clean.shape, adata_imgl_clean.shape

In [ ]:
print(mdata_dict['cite_6tf'].shape, mdata_dict['cite_imgl'].shape)
mdata_6tf_clean = mdata_dict['cite_6tf'][~mdata_dict['cite_6tf'].mod['rna'].obs['guide'].isin(["non-targeting_g5", "FOXK1_g2"])]
mdata_imgl_clean = mdata_dict['cite_imgl'][~mdata_dict['cite_imgl'].mod['rna'].obs['guide'].isin(["non-targeting_g5", "FOXK1_g2"])]
mdata_6tf_clean.shape, mdata_imgl_clean.shape

In [ ]:
mixscale_col = "mixscale_score"

In [ ]:
adata_6tf_masked = adata_6tf_clean[adata_6tf_clean.obs[mixscale_col] >= 0].copy()
adata_imgl_masked = adata_imgl_clean[adata_imgl_clean.obs[mixscale_col] >= 0].copy()
adata_6tf_masked.shape, adata_imgl_masked.shape

In [ ]:
adata_masked_dict = {}
adata_masked_dict['iTF'] = adata_6tf_masked
adata_masked_dict['iMG'] = adata_imgl_masked

# Genes to Include

In [ ]:
genes_to_include = ['DNMT1', 'IRF9', 'STAT2', 'SMAD3', 'PRDM1', 'ZNF532', 'NTC']
itf_genes = ['DNMT1', 'IRF9', 'STAT2', 'SMAD3']
img_genes = ['PRDM1', 'ZNF532']

# Save Directory

In [ ]:
save_dir = "<path to output data directory>"
fig_dir = "<path to output factor directory>"

# Load in Percentile Shifts

In [ ]:
percentile_df_6tf_fact_gene = pd.read_csv(save_dir + "ps_iTF_factor_gene_v5.csv")
percentile_df_6tf_fact_guide = pd.read_csv(save_dir + "ps_iTF_factor_guide_v5.csv")
percentile_df_img_fact_gene = pd.read_csv(save_dir + "ps_iMG_factor_gene_v5.csv")
percentile_df_img_fact_guide = pd.read_csv(save_dir + "ps_iMG_factor_guide_v5.csv")

In [ ]:
percentile_df_6tf_sign_gene = pd.read_csv(save_dir + "ps_iTF_signature_gene_v5.csv")
percentile_df_6tf_sign_guide = pd.read_csv(save_dir + "ps_iTF_signature_guide_v5.csv")
percentile_df_img_sign_gene = pd.read_csv(save_dir + "ps_iMG_signature_gene_v5.csv")
percentile_df_img_sign_guide = pd.read_csv(save_dir + "ps_iMG_signature_guide_v5.csv")

In [ ]:
percentile_df_6tf_fact_gene["sig"] = percentile_df_6tf_fact_gene["sig"].replace(np.NaN, "")
percentile_df_6tf_fact_guide["sig"] = percentile_df_6tf_fact_guide["sig"].replace(np.NaN, "")
percentile_df_img_fact_gene["sig"] = percentile_df_img_fact_gene["sig"].replace(np.NaN, "")
percentile_df_img_fact_guide["sig"] = percentile_df_img_fact_guide["sig"].replace(np.NaN, "")



In [ ]:
percentile_df_6tf_sign_gene["sig"] = percentile_df_6tf_sign_gene["sig"].replace(np.NaN, "")
percentile_df_6tf_sign_guide["sig"] = percentile_df_6tf_sign_guide["sig"].replace(np.NaN, "")
percentile_df_img_sign_gene["sig"] = percentile_df_img_sign_gene["sig"].replace(np.NaN, "")
percentile_df_img_sign_guide["sig"] = percentile_df_img_sign_guide["sig"].replace(np.NaN, "")


# Visualize Factor Point Plots

In [ ]:
factors_to_include = ['apoe-high',
                     'ciita-high',
                     'c1q-high/phagocytic',
                     'chemokine',
                     'cx3cr1-high',
                     'gpnmb-high',
                     'hla-high/apc',
                     'ifn-i response',
                     'immunoregulatory',
                     'motility/adhesion',
                     'npy1r-high',
                     'oxphos-1',
                     'plcg2-high',
                     's100/tlr signaling',
                     'senescence',
                     'tlr/mapk signaling',
                     'stress']

In [ ]:
percentile_df_img_fact_guide['perturbed_gene'] = percentile_df_img_fact_guide['perturbed_guide'].str.split("_").str[0]

In [ ]:
factor_cols = adata_6tf_masked.obs.columns[(adata_6tf_masked.obs.columns.str.startswith("f")) &
                                           ~(adata_6tf_masked.obs.columns.isin(['f6','f12', 'f13']))]

In [ ]:
factor_cols_labels = [factor_labels.get_direct_factor_map()[f] for f in factor_cols]

In [ ]:
factor_order = percentile_df_img_fact_guide[percentile_df_img_fact_guide['perturbed_guide']== "PRDM1_g1"].sort_values(by="PctShift")['Factor'].tolist()

factor_labels.draw_pointplot(percentile_df_img_fact_guide,
                ["PRDM1"],
                (12,10),
                by_guide=True,
                diff_type_name="Factor",
                nrows=1,
                ncols=2,
                palette=factor_labels.palette_dict_imgl,
                orderlist=factor_order,
                filename="figures/iMG/PRDM1_percentile_shift_factor.svg"
            )
factor_order = percentile_df_img_fact_guide[percentile_df_img_fact_guide['perturbed_guide']== "ZNF532_g2"].sort_values(by="PctShift")['Factor'].tolist()

factor_labels.draw_pointplot(percentile_df_img_fact_guide,
                ["ZNF532"],
                (12,10),
                by_guide=True,
                diff_type_name="Factor",
                nrows=1,
                ncols=2,
                palette=factor_labels.palette_dict_imgl,
                orderlist=factor_order,
                filename="figures/iMG/ZNF532_percentile_shift_factor.svg"
            )

In [ ]:
factor_order = percentile_df_6tf_fact_guide[percentile_df_6tf_fact_guide['perturbed_guide']== "STAT2_g2"].sort_values(by="PctShift")['Factor'].tolist()

In [ ]:
factor_order = percentile_df_6tf_fact_guide[percentile_df_6tf_fact_guide['perturbed_guide']== "STAT2_g2"].sort_values(by="PctShift")['Factor'].tolist()
factor_labels.draw_pointplot(percentile_df_6tf_fact_guide,
                ["STAT2"],
                (12,10),
                by_guide=True,
                diff_type_name="Factor",
                nrows=1,
                ncols=2,
                palette=factor_labels.palette_dict_6tf,
                orderlist=factor_order,
                filename="figures/iTF/STAT2_percentile_shift_factor.svg"
            )

factor_order = percentile_df_6tf_fact_guide[percentile_df_6tf_fact_guide['perturbed_guide']== "DNMT1_g2"].sort_values(by="PctShift")['Factor'].tolist()
factor_labels.draw_pointplot(percentile_df_6tf_fact_guide,
                ["DNMT1"],
                (12,10),
                by_guide=True,
                diff_type_name="Factor",
                nrows=1,
                ncols=2,
                palette=factor_labels.palette_dict_6tf,
                orderlist=factor_order,
                filename="figures/iTF/DNMT1_percentile_shift_factor.svg"
            )

# Visualize NTC Radar Plots

In [ ]:
img_fdf = adata_imgl_masked.obs[adata_imgl_masked.obs['perturbed_gene'] == "NTC"][factor_cols]
itf_fdf = adata_6tf_masked.obs[adata_6tf_masked.obs['perturbed_gene'] == "NTC"][factor_cols]

In [ ]:
itf_fdf_med = pd.DataFrame(itf_fdf.median())
itf_fdf_med = itf_fdf_med.rename(columns={0:"MedianFactor"})
itf_fdf_med=itf_fdf_med.sort_values(by="MedianFactor")
itf_fdf_med_T = itf_fdf_med.T

In [ ]:
img_fdf_med = pd.DataFrame(img_fdf.median())
img_fdf_med = img_fdf_med.rename(columns={0:"MedianFactor"})
img_fdf_med=img_fdf_med.sort_values(by="MedianFactor")
img_fdf_med_T = img_fdf_med.T

In [ ]:
fig = factor_labels.plot_radar_chart(itf_fdf_med_T, factor_cols, title="iTF-MG NTC Distribution Across Factors")
plt.savefig("figures/iTF/final/itf_factor_radar.svg")
plt.show()


In [ ]:
fig = factor_labels.plot_radar_chart(img_fdf_med_T, factor_cols, title="iMG NTC Distribution Across Factors")
plt.savefig("figures/iMG/final/img_ntc_factor_radar.svg")
plt.show()